# OBJECTIVE

# Refactor the delivery-time prediction code built in P02/P03 from scattered notebook cells into a proper Python package (delivery/) with separate modules — data.py, features.py, model.py, and validate.py — tied together with an __init__.py. Import and use this package like a library to train and save a model, then run the full pipeline from the command line using standalone scripts (train.py and predict.py), without relying on a notebook.

# Tasks:

# T1: Add a function average_speed_kmph(distance_km, delivery_min) to the package.

# T2: Add a delivery/validate.py module with a function that rejects an impossible/unrealistic order.

# T3: Write a second command-line script, predict.py, that loads the saved model and predicts delivery time for a new order.

# Create Folders

In [ ]:
#`delivery/` will hold our package files, `data/` will hold the CSV.

In [ ]:
import os
os.makedirs("delivery", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("Folders ready")

Folders ready


# Step 1: data.py --- loads the data

In [ ]:
%%writefile delivery/data.py
import pandas as pd
import numpy as np
import os

def load_data():
    path = "data/delivery_times.csv"

    if not os.path.exists(path):
        np.random.seed(42)
        n = 600
        distance_km = np.random.uniform(0.5, 12, n)
        prep_time_min = np.random.uniform(5, 30, n)
        traffic_level = np.random.randint(1, 4, n)
        rain = np.random.randint(0, 2, n)
        noise = np.random.normal(0, 2, n)

        delivery_min = 6 + 3 * distance_km + 0.6 * prep_time_min + 4 * traffic_level + 5 * rain + noise

        df = pd.DataFrame({
            "distance_km": distance_km,
            "prep_time_min": prep_time_min,
            "traffic_level": traffic_level,
            "rain": rain,
            "delivery_min": delivery_min
        })
        df.to_csv(path, index=False)

    return pd.read_csv(path)

Overwriting delivery/data.py


# Step 2: features.py --- splits X and y (also has T1)

# Splits the data into inputs (X) and the target to predict (y).
# average_speed_kmph is Task T1.

In [ ]:
%%writefile delivery/features.py
def get_features_and_target(df):
    X = df[["distance_km", "prep_time_min", "traffic_level", "rain"]]
    y = df["delivery_min"]
    return X, y

# T1
def average_speed_kmph(distance_km, delivery_min):
    hours = delivery_min / 60
    return distance_km / hours

Overwriting delivery/features.py


# Step 3: model.py --- trains, checks, and saves the model

# Trains a LinearRegression model, prints its test error (MAE), and saves it to a file so predict.py can reuse it later.

In [ ]:
%%writefile delivery/model.py
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

def train_and_save_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    print("Test MAE:", round(mae, 2))

    joblib.dump(model, "delivery_model.joblib")
    return model

Overwriting delivery/model.py


# Step 4: validate.py --- T2, rejects a bad order
# Checks that an order's values are realistic before we predict on it.

In [ ]:
%%writefile delivery/validate.py
# T2
def is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    if distance_km <= 0:
        return False
    if prep_time_min <= 0:
        return False
    if traffic_level not in [1, 2, 3]:
        return False
    if rain not in [0, 1]:
        return False
    return True

Overwriting delivery/validate.py


# Step 5: __init__.py --- makes delivery a package

# Built last so it can import functions from all the other files. This is what makes `from delivery import ...` work.

In [ ]:
%%writefile delivery/__init__.py
from .data import load_data
from .features import get_features_and_target, average_speed_kmph
from .model import train_and_save_model
from .validate import is_valid_order

Overwriting delivery/__init__.py


# Step 6: Import and use our own package
# Proves the package works --- import it like a library and train the model.

In [ ]:
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)

Test MAE: 1.92


# Step 7: train.py  script

In [ ]:
%%writefile train.py
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)
print("Training done")

Overwriting train.py


# Step 8: Run train.py like a real program
# Runs the script as a terminal command, no notebook involved.

In [ ]:
!python train.py

Test MAE: 1.92
Training done


# T3: predict.py --- second script
# Loads the saved model and predicts delivery time for one new order.

In [ ]:
%%writefile predict.py
import sys
import joblib
import pandas as pd
from delivery import is_valid_order

distance_km = float(sys.argv[1])
prep_time_min = float(sys.argv[2])
traffic_level = int(sys.argv[3])
rain = int(sys.argv[4])

if not is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    print("Invalid order")
else:
    model = joblib.load("delivery_model.joblib")
    order = pd.DataFrame([[distance_km, prep_time_min, traffic_level, rain]],
                          columns=["distance_km", "prep_time_min", "traffic_level", "rain"])
    prediction = model.predict(order)[0]
    print("Predicted delivery time:", round(prediction, 1), "minutes")

Overwriting predict.py


# Test predict.py --- a normal order

In [ ]:
!python predict.py 5.0 15 2 0

Predicted delivery time: 39.8 minutes


# Test predict.py --- a bad order (T2 check)

In [ ]:
!python predict.py -3 15 2 0

Invalid order


# Test T1: average_speed_kmph

In [ ]:
from delivery import average_speed_kmph

speed = average_speed_kmph(distance_km=5.0, delivery_min=40.4)
print("Average speed (km/h):", round(speed, 2))

Average speed (km/h): 7.43


# Summary


- data.py --- loads the delivery data
- features.py --- splits data into X and y
- model.py --- trains, checks, and saves the model
- validate.py --- rejects a bad order
- __init__.py --- makes the folder a package

# To Do: Package the classification workflow

Using the same pattern from this lab (data.py -> features.py -> model.py ->
validate.py -> __init__.py), turn the classification code from Lab 3
(Breast Cancer dataset) into its own package. This time, go a step further
than just training one fixed model.

T1 --- model.py should not train just one model. Use cross-validation to
compare LogisticRegression, DecisionTreeClassifier, and
RandomForestClassifier, and automatically save whichever one scores best
(instead of hardcoding the winner yourself).

T2 --- validate.py should check that at least 3 of the input measurements
fall within a realistic range (e.g. radius_mean and area_mean can't be
negative or absurdly large) --- not just "is this a number".

T3 --- predict.py should print both the prediction (malignant/benign) AND
the model's confidence for that prediction (use predict_proba).

T4 --- add a metrics.py module with one function that prints a
confusion matrix and classification report for the saved model, so anyone
can check its performance without retraining.

In [1]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

os.makedirs("breast_cancer", exist_ok=True)

df = pd.read_csv("breast-cancer.csv")

df = df.drop(columns=["id"], errors="ignore")
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"].map({"M": 1, "B": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000))
    ]),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42)
}

scores = {}

for name, model in models.items():
    score = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy").mean()
    scores[name] = score
    print(name, "CV Accuracy:", round(score, 4))

best_model_name = max(scores, key=scores.get)
best_model = models[best_model_name]

best_model.fit(X_train, y_train)
joblib.dump(best_model, "breast_cancer/best_model.joblib")

depths = [2, 3, 4, 5, 6, 8, 10, None]
depth_scores = {}

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    score = cross_val_score(
        tree, X_train, y_train, cv=5, scoring="accuracy"
    ).mean()
    depth_scores[depth] = score

best_depth = max(depth_scores, key=depth_scores.get)

print("\nBest Model:", best_model_name)
print("Best Tree Depth:", best_depth)

def is_valid_measurements(row):
    checks = [
        0 < row["radius_mean"] < 50,
        0 < row["texture_mean"] < 100,
        0 < row["perimeter_mean"] < 300,
        0 < row["area_mean"] < 5000,
        0 < row["smoothness_mean"] < 1
    ]
    return sum(checks) >= 3

def print_metrics(model, X_test, y_test):
    predictions = model.predict(X_test)
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, predictions))
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        predictions,
        target_names=["Benign", "Malignant"]
    ))

print_metrics(best_model, X_test, y_test)

sample = X_test.iloc[0]

print("\nSample Valid:", is_valid_measurements(sample))

prediction = best_model.predict([sample])[0]
probability = best_model.predict_proba([sample])[0].max()

print("Prediction:", "Malignant" if prediction == 1 else "Benign")
print("Confidence:", round(probability * 100, 2), "%")

LogisticRegression CV Accuracy: 0.9714
DecisionTree CV Accuracy: 0.9319
RandomForest CV Accuracy: 0.9626

Best Model: LogisticRegression
Best Tree Depth: 5

Confusion Matrix:
[[71  1]
 [ 3 39]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      0.99      0.97        72
   Malignant       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114


Sample Valid: True
Prediction: Benign
Confidence: 99.96 %


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
